# 03 PyTorch Training v1

Persistent PyTorch warm-up + fine-tune test run. This notebook uses the existing NB02 split manifests and `src_torch` helpers; TensorFlow notebooks are not modified.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('MPLCONFIGDIR', '/private/tmp')

print('PROJECT_ROOT:', PROJECT_ROOT)

PROJECT_ROOT: /Users/starsrain/2025_codeProject/GreenSpace_CNN


In [2]:
import torch
import torchvision
import torchgeo

from src_torch.config import TORCH_DATA_CONFIG, TORCH_MODEL_CONFIG, TORCH_TRAINING_CONFIG
from src_torch.training import run_persistent_warmup_finetune

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('torchgeo:', torchgeo.__version__)
print('TORCH_DATA_CONFIG:', TORCH_DATA_CONFIG)
print('TORCH_MODEL_CONFIG:', TORCH_MODEL_CONFIG)
print('TORCH_TRAINING_CONFIG:', TORCH_TRAINING_CONFIG)

torch: 2.10.0
torchvision: 0.25.0
torchgeo: 0.8.1
TORCH_DATA_CONFIG: {'img_size': (512, 512), 'batch_size': 4, 'num_workers': 0, 'pin_memory': False, 'image_transform': 'tf_parity', 'backbone_preprocess': None, 'use_oversampling': True, 'use_augmentation': True, 'oversampling_sanity_batches': 80}
TORCH_MODEL_CONFIG: {'backbone_priority': 'torchgeo', 'torchgeo_model_name': 'swin_v2_b', 'torchgeo_weight': 'Swin_V2_B_Weights.NAIP_RGB_SI_SATLAS', 'load_pretrained_weights': True, 'preserve_input_resolution': True, 'fallback_backbone': 'torchvision', 'num_binary': 7, 'num_shade': 2, 'score_output_range': (1.0, 5.0), 'veg_output_range': (1.0, 5.0)}
TORCH_TRAINING_CONFIG: {'seed': 37, 'test_run_mode': True, 'test_warmup_epochs': 1, 'test_finetune_epochs': 1, 'warmup_epochs': 5, 'finetune_epochs': 15, 'warmup_learning_rate': 0.001, 'finetune_learning_rate': 0.0001, 'fine_tune_backbone': True, 'use_combo_training_control': True, 'combo_w_bin': 0.5, 'combo_w_ord': 0.5, 'combo_mcmae_scale': 2.0, '

## Run 1+1 Training Test

Expected behavior: one warm-up epoch with the TorchGeo backbone frozen, then one fine-tune epoch with the backbone trainable when `fine_tune_backbone=True`. Artifacts are saved under `models/runs/PyTorch_<timestamp>/`.

In [3]:
# Full PyTorch training run. `auto` selects CUDA, MPS, or CPU.
training_override = {
    'test_run_mode': False,
    'device': 'auto',
    'max_train_batches': None,
    'max_val_batches': None,
}

result = run_persistent_warmup_finetune(training_config=training_override)
result

Matplotlib is building the font cache; this may take a moment.


RUN_TAG: PyTorch_20260614_220926
RUN_DIR: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926
device: mps
test_run_mode=False -> warmup=5, finetune=15
batch caps: train=None, val=None
model metadata: {'model_name': 'swin_v2_b', 'weight_name': 'Swin_V2_B_Weights.NAIP_RGB_SI_SATLAS', 'load_pretrained_weights': True, 'preserve_input_resolution': True, 'requested_input_size': (512, 512), 'feature_dim': 1024, 'weight_meta': {'dataset': 'SatlasPretrain', 'in_chans': 3, 'model': 'swin_v2_b', 'publication': 'https://arxiv.org/abs/2211.15660', 'repo': 'https://github.com/allenai/satlas', 'bands': ('R', 'G', 'B')}, 'weight_transforms': 'Sequential(\n  (0): CenterCrop(size=(256, 256))\n  (1): Normalize(mean=[0], std=[255], inplace=True)\n)', 'official_weight_transforms': 'Sequential(\n  (0): CenterCrop(size=(256, 256))\n  (1): Normalize(mean=[0], std=[255], inplace=True)\n)', 'effective_weight_transforms': 'Sequential(\n  (0): Normalize(mean=[0], std=[255], inplac

{'run_tag': 'PyTorch_20260614_220926',
 'run_dir': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926',
 'final_model_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926/final_PyTorch_20260614_220926.pt',
 'best_mc_mae_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926/best_mcmae_PyTorch_20260614_220926.pt',
 'best_prauc_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926/best_prauc_PyTorch_20260614_220926.pt',
 'model_config_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926/model_config_PyTorch_20260614_220926.json',
 'training_history_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926/training_history_PyTorch_20260614_220926.json',
 'training_curves_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926/tra

In [4]:
run_dir = Path(result['run_dir'])
print('RUN_DIR:', run_dir)
for path in sorted(run_dir.iterdir()):
    print(path.name)

RUN_DIR: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926
best_mcmae_PyTorch_20260614_220926.pt
best_prauc_PyTorch_20260614_220926.pt
final_PyTorch_20260614_220926.pt
model_config_PyTorch_20260614_220926.json
training_curves.png
training_history_PyTorch_20260614_220926.json
